## An example of using Jupyter for Documenting and Automating BDD Style Tests

### Given the Playwright background worker is running

In [ ]:
import threading
import sys
import asyncio
import queue
from playwright.sync_api import sync_playwright

task_queue = queue.Queue()
pw_state = {}
error_state = {'error': None}

def playwright_worker():
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    pw = sync_playwright().start()
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    
    pw_state['pw'] = pw
    pw_state['browser'] = browser
    pw_state['page'] = page
    
    while True:
        task = task_queue.get()
        if task is None:
            break
        try:
            task(pw_state)
        except Exception as e:
            error_state['error'] = e
        task_queue.task_done()
        
    browser.close()
    pw.stop()

worker_thread = threading.Thread(target=playwright_worker, daemon=True)
worker_thread.start()

def execute(func):
    error_state['error'] = None
    task_queue.put(func)
    task_queue.join()
    if error_state['error']:
        raise error_state['error']

### When I navigate to Google

In [ ]:
def go_to_google(state):
    page = state['page']
    page.goto('https://www.google.com')
    try:
        page.click('button:has-text("Accept all")', timeout=2000)
    except:
        pass
    
    assert 'Google' in page.title(), f"Expected 'Google', got {page.title()}"
    
execute(go_to_google)

### Then I can search for genuflect

In [ ]:
def search_term(state):
    page = state['page']
    search_box = page.locator("textarea[name='q'], input[name='q']").first
    search_box.fill("genuflect")
    page.keyboard.press("Enter")
    
    page.wait_for_load_state("networkidle")
    
    assert 'genuflect' in page.title().lower(), "Search did not complete successfully"
    
execute(search_term)

### And the browser shuts down cleanly

In [ ]:
task_queue.put(None)
worker_thread.join(timeout=10)